In [0]:
# ==========================================
# Project: Retail Medallion Architecture
# Layer: Gold
# Purpose: Country Revenue KPI
# ==========================================

from pyspark.sql import functions as F

# ------------------------------------------
# Configuration
# ------------------------------------------

SILVER_TABLE = "retail_project.silver.online_retail"
GOLD_TABLE = "retail_project.gold.country_sales"

# ------------------------------------------
# Read Silver Data
# ------------------------------------------

df_silver = spark.table(SILVER_TABLE)

# ------------------------------------------
# Revenue Calculation
# ------------------------------------------

df_gold_base = (
    df_silver
    .withColumn(
        "revenue",
        F.round(
            F.col("Quantity") * F.col("UnitPrice"),
            2
        )
    )
)

# ------------------------------------------
# Country Aggregation
# ------------------------------------------

country_sales = (
    df_gold_base
    .groupBy("Country")
    .agg(
        F.round(
            F.sum("revenue"),
            2
        ).alias("total_revenue")
    )
    .orderBy(F.desc("total_revenue"))
)

# ------------------------------------------
# Validation
# ------------------------------------------

country_sales.show(20, False)

# ------------------------------------------
# Write Gold Table
# ------------------------------------------

(
    country_sales.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_TABLE)
)

print("Country Sales Gold table created successfully.")